## LULC Temporal Stability

We can't do this yet because we only have 2 time points of LULC. I could run a single tile for all years as a test. This is a plan to assess temporal stability across LULC outputs.

For each tile:
1. Load all LULC assets for the 2000-2025 time range
2. Flip rate: count how many times each pixel changes. Flag pixels over a threshhold.
3. Reversals/Flicker: Count A→B→A patterns within 1–3 years, such as Tree→Grassland→Tree. These are almost always classification noise rather than real change, which makes this the most diagnostic single metric. Flag flickering pixels and note which classes over which years.
4. Implausible transitions. Build an annual transition matrix and flag transitions that are unlikely within one year, such as Built-up→Tree cover, Water→Built-up outside known reclamation, or Mangrove↔Cropland. Report their share of all change.
5. Temporal stability. For each pixel, record the percentage of years matching its modal class and the number of distinct classes it takes over 26 years. Map these to find unstable hotspots, which typically fall in coastal mixed pixels, cloud-prone areas and grass/shrub boundaries.
6. Class area time series. Plot annual area per class. Year-to-year jumps that coincide with input changes are artefacts, not land change. Common causes are Landsat 5→7→8/9 sensor transitions, Landsat 7 SLC-off gaps after 2003, and years with low clear-observation counts in the GeoMAD.
7. Invariant areas. Areas known to be stable, such as protected forest or old urban cores, should show close to zero change.

TODO: What do do if we see bad temporal stability for many pixels, especially for certain classes.

In [ ]:
# Reload functions during development
%load_ext autoreload
%autoreload 2

In [ ]:
# TODO: Run all LULC for this tile.
# TODO: Run this for all test tiles? Not just 1 other tile.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from odc.stac import load
from pystac import Item
from rustac import search_sync

from ldn.aws import configure_s3_access_profile
from ldn.typology import classes, classes_flipped, colors
from ldn.utils import (
    LULC_DATASET_ID,
    SENSOR,
    dataset_prefix,
)

configure_s3_access_profile()

In [ ]:
BUCKET_URL = "https://dep-public-staging.s3.us-west-2.amazonaws.com"
LULC_VERSION = "0-0-9"
OWNER = "dep"
LULC_PREFIX = dataset_prefix(OWNER, SENSOR, LULC_DATASET_ID)
LULC_PARQUET = f"{BUCKET_URL}/{LULC_PREFIX}/{LULC_VERSION}/{LULC_PREFIX}.parquet"
REPORT_DIR = Path("reports_temporal") / LULC_VERSION  # Relative to this notebook (notebooks/LULC_Testing)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
TILES = [(63, 20)]  # SW Viti Levu, Fiji
YEARS = range(2000, 2026)
NODATA = 255

FLIP_THRESHOLD = 4  # flag pixels with more than this many year-to-year changes
FLICKER_MAX_YEARS = 3  # A -> B -> A where B lasts at most this many years
MODAL_THRESHOLD = 0.8  # flag pixels whose modal class covers less than this share of years
DISTINCT_THRESHOLD = 3  # flag pixels with more than this many distinct classes

# Transitions that are unlikely within one year
IMPLAUSIBLE = [
    (classes["Built-up"], classes[label]) for label in ["Tree cover", "Grassland", "Cropland", "Wetland", "Water"]
] + [
    (classes["Water"], classes["Built-up"]),
    (classes["Wetland"], classes["Cropland"]),
    (classes["Cropland"], classes["Wetland"]),
]

In [ ]:
def load_tile(x: int, y: int):
    """Load the annual LULC classification for one tile as a (time, y, x) DataArray."""
    ids = [f"{LULC_PREFIX}_{x:03}_{y:03}_{year}" for year in YEARS]
    docs = search_sync(LULC_PARQUET, ids=ids)
    items = [Item.from_dict(doc) for doc in docs]
    # TODO: Reenable the check for missing items once the data is complete.
    # if len(items) != len(ids):
    #     raise ValueError(f"Found {len(items)} of {len(ids)} items for tile {x:03}_{y:03}")
    return load(items, measurements=["classification"], chunks={})["classification"].load()


def class_counts(a: np.ndarray) -> np.ndarray:
    """Count the years each pixel is in each class, as an array of shape (class, y, x)."""
    return np.stack([(a == c).sum(axis=0) for c in classes_flipped if c != NODATA])


def flip_rate(a: np.ndarray) -> np.ndarray:
    """Flag pixels with more than FLIP_THRESHOLD year-to-year changes."""
    return (a[1:] != a[:-1]).sum(axis=0) > FLIP_THRESHOLD


def flicker_events(a: np.ndarray) -> np.ndarray:
    """Find A -> B -> A patterns where B lasts at most FLICKER_MAX_YEARS years.

    Each event is marked once, at the year A changes to B, using the first year the pixel returns to A.

    Args:
        a: Class array of shape (time, y, x).

    Returns:
        Boolean array of shape (time - 1, y, x).
    """
    unreturned = a[1:] != a[:-1]
    events = np.zeros_like(unreturned)
    for lag in range(2, FLICKER_MAX_YEARS + 2):
        returned = unreturned[: len(a) - lag] & (a[:-lag] == a[lag:])
        unreturned[: len(a) - lag] &= ~returned
        events[: len(a) - lag] |= returned
    return events


def flicker(events: np.ndarray) -> np.ndarray:
    """Flag pixels with any flicker event from flicker_events."""
    return events.any(axis=0)


def implausible_events(a: np.ndarray) -> np.ndarray:
    """Find year-to-year transitions in IMPLAUSIBLE.

    Args:
        a: Class array of shape (time, y, x).

    Returns:
        Boolean array of shape (time - 1, y, x), True at the year each transition starts.
    """
    lookup = np.zeros((256, 256), bool)
    for src, dst in IMPLAUSIBLE:
        lookup[src, dst] = True
    return lookup[a[:-1], a[1:]]


def implausible(events: np.ndarray) -> np.ndarray:
    """Flag pixels with any implausible transition from implausible_events."""
    return events.any(axis=0)


def low_modal_share(a: np.ndarray) -> np.ndarray:
    """Flag pixels whose modal class covers less than MODAL_THRESHOLD of years."""
    return class_counts(a).max(axis=0) / len(a) < MODAL_THRESHOLD


def many_classes(a: np.ndarray) -> np.ndarray:
    """Flag pixels with more than DISTINCT_THRESHOLD distinct classes."""
    return (class_counts(a) > 0).sum(axis=0) > DISTINCT_THRESHOLD


def flag_pixels(a: np.ndarray, flickers: np.ndarray, implausibles: np.ndarray) -> dict:
    """Flag temporally unstable pixels for each test.

    Args:
        a: Class array of shape (time, y, x).
        flickers: Flicker events from flicker_events.
        implausibles: Implausible transitions from implausible_events.

    Returns:
        Dict of boolean (y, x) arrays: "valid" plus one per test. Pixels with nodata in any year are
        not valid and are never flagged.
    """
    valid = (a != NODATA).all(axis=0)
    flags = {
        "flip_rate": flip_rate(a),
        "flicker": flicker(flickers),
        "implausible": implausible(implausibles),
        "low_modal_share": low_modal_share(a),
        "many_classes": many_classes(a),
    }
    flags = {name: flagged & valid for name, flagged in flags.items()}
    flags["any"] = np.logical_or.reduce(list(flags.values()))
    flags["valid"] = valid
    return flags


def categorise(codes: np.ndarray, valid: np.ndarray, known: np.ndarray | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Categorise each pixel by the codes it has.

    Args:
        codes: Integer array of shape (n, y, x), -1 where there is no code.
        valid: Boolean (y, x) array of valid pixels.
        known: All possible codes, to list codes with no pixels. If None, only codes found are listed.

    Returns:
        (y, x) category array and the codes it refers to. Category 0 is no code, 1 to n is the matching
        returned code, n + 1 is more than one code and n + 2 is not valid.
    """
    first = np.where(codes >= 0, codes, np.iinfo(codes.dtype).max).min(axis=0)
    last = codes.max(axis=0)
    single = first == last
    unique = np.unique(last[single & valid] if known is None else known)
    category = np.select(
        [~valid, last < 0, ~single],
        [len(unique) + 2, 0, len(unique) + 1],
        np.searchsorted(unique, last) + 1,
    )
    return category, unique


def transition_categories(
    a: np.ndarray, events: np.ndarray, valid: np.ndarray, known: list[tuple[int, int]] | None = None
) -> tuple[np.ndarray, np.ndarray]:
    """Categorise each pixel by the year-to-year transitions marked in events.

    Args:
        a: Class array of shape (time, y, x).
        events: Boolean array of shape (time - 1, y, x) marking the transitions to use.
        valid: Boolean (y, x) array of valid pixels.
        known: All possible (src, dst) transitions. If None, only transitions found are listed.

    Returns:
        Category array and codes as from categorise. Transitions are encoded as src * 256 + dst.
    """
    codes = np.where(events, a[:-1].astype(np.int32) * 256 + a[1:], -1)
    return categorise(codes, valid, None if known is None else np.array([src * 256 + dst for src, dst in known]))


def plot_categories(category: np.ndarray, labels: list[str], title: str, png: Path):
    """Map categories with the pixel count and share of each category in the legend.

    Args:
        category: (y, x) category array from categorise.
        labels: Label for each category: none, one per code, multiple, nodata.
        title: Plot title.
        png: Path to save the figure to.
    """
    # ponytail: colours repeat after 50 codes, group rare codes into "Other" if tiles get that busy
    palette = [*plt.get_cmap("tab10").colors, *plt.get_cmap("tab20b").colors, *plt.get_cmap("tab20c").colors]
    colours = ["white", *(palette[i % len(palette)] for i in range(len(labels) - 3)), "black", "lightgrey"]
    counts = np.bincount(category.ravel(), minlength=len(labels))
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.imshow(category, cmap=ListedColormap(colours), vmin=0, vmax=len(labels) - 1, interpolation="nearest")
    ax.set_title(title)
    fig.legend(
        handles=[
            Patch(facecolor=colour, edgecolor="black", label=f"{label}: {count:,} ({count / category.size:.1%})")
            for colour, label, count in zip(colours, labels, counts)
        ],
        loc="center left",
        bbox_to_anchor=(1, 0.5),
    )
    fig.savefig(png, dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
rows = {}
flags_by_tile = {}
flicker_by_tile = {}
implausible_by_tile = {}
areas_by_tile = {}  # TODO: 3832 is not an equal area crs. Reproject to 6933? Will break antimeridian.
years_by_tile = {}

for x, y in TILES:
    lulc = load_tile(x, y)
    flickers = flicker_events(lulc.values)
    implausibles = implausible_events(lulc.values)
    flags = flag_pixels(lulc.values, flickers, implausibles)
    flags_by_tile[(x, y)] = flags
    flicker_by_tile[(x, y)] = transition_categories(lulc.values, flickers, flags["valid"])
    implausible_by_tile[(x, y)] = transition_categories(lulc.values, implausibles, flags["valid"], IMPLAUSIBLE)
    years_by_tile[(x, y)] = lulc.time.dt.year.values.tolist()
    rows[f"{x:03}_{y:03}"] = {k: int(v.sum()) for k, v in flags.items()}

    res = lulc.odc.geobox.resolution
    pixel_km2 = abs(res.x * res.y) / 1e6
    areas_by_tile[(x, y)] = pd.DataFrame(
        {
            classes_flipped[c]: (lulc == c).sum(dim=["x", "y"]).values * pixel_km2
            for c in classes_flipped
            if c != NODATA
        },
        index=lulc.time.dt.year.values,
    )

summary = pd.DataFrame.from_dict(rows, orient="index")
summary.index.name = "tile"
summary

Pixels not temporally stable per tile, by test. A pixel can fail several tests, so the test columns do not sum to `any`.
Pixels with nodata in any year are excluded (`valid` is the number of pixels tested).

In [ ]:
pct = (summary.drop(columns="valid").div(summary["valid"], axis=0) * 100).round(1)
pct

In [ ]:
# Map of unstable pixels per tile by test, to find hotspots
tests = summary.columns.drop(["any", "valid"])
unstable_pngs = {}
for tile, flags in flags_by_tile.items():
    codes = np.where(np.stack([flags[name] for name in tests]), np.arange(len(tests))[:, None, None], -1)
    category, found = categorise(codes, flags["valid"], np.arange(len(tests)))
    unstable_pngs[tile] = REPORT_DIR / f"unstable_pixels_{tile[0]:03}_{tile[1]:03}.png"
    labels = ["Stable", *tests[found], "Multiple tests", "Nodata"]
    plot_categories(category, labels, f"Unstable pixels, tile {tile[0]:03}_{tile[1]:03}", unstable_pngs[tile])

In [ ]:
# Map of flicker class per tile, to see which classes the model confuses
flicker_pngs = {}
for tile, (category, found) in flicker_by_tile.items():
    pairs = [(classes_flipped[code // 256], classes_flipped[code % 256]) for code in found]
    labels = ["No flicker", *[f"{src} -> {mid} -> {src}" for src, mid in pairs], "Multiple", "Nodata"]
    flicker_pngs[tile] = REPORT_DIR / f"flicker_classes_{tile[0]:03}_{tile[1]:03}.png"
    plot_categories(category, labels, f"Flicker classes, tile {tile[0]:03}_{tile[1]:03}", flicker_pngs[tile])

In [ ]:
# Map of implausible transitions per tile
implausible_pngs = {}
for tile, (category, found) in implausible_by_tile.items():
    pairs = [(classes_flipped[code // 256], classes_flipped[code % 256]) for code in found]
    labels = ["None", *[f"{src} -> {dst}" for src, dst in pairs], "Multiple", "Nodata"]
    implausible_pngs[tile] = REPORT_DIR / f"implausible_transitions_{tile[0]:03}_{tile[1]:03}.png"
    plot_categories(
        category, labels, f"Implausible transitions, tile {tile[0]:03}_{tile[1]:03}", implausible_pngs[tile]
    )

In [ ]:
# Class area time series. Look for jumps that coincide with Landsat sensor changes or GeoMAD observation counts.
area_pngs = {}
for tile, areas in areas_by_tile.items():
    # Typology colours are "rgb(r,g,b)" strings, matplotlib needs 0-1 tuples
    area_colours = [tuple(int(v) / 255 for v in colors[classes[label]][4:-1].split(",")) for label in areas]
    ax = areas.plot(
        figsize=(12, 5),
        title=f"Class area, tile {tile[0]:03}_{tile[1]:03}",
        ylabel="Area (km2)",
        color=area_colours,
    )
    ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))
    area_pngs[tile] = REPORT_DIR / f"class_area_{tile[0]:03}_{tile[1]:03}.png"
    ax.figure.savefig(area_pngs[tile], dpi=150, bbox_inches="tight")

## Write the report

In [ ]:
def markdown_table(df: pd.DataFrame) -> str:
    """Render a DataFrame as a markdown table.

    Args:
        df: Table to render. The index is written as columns.

    Returns:
        Markdown table string.
    """
    df = df.reset_index().rename(columns={"index": ""})
    rows = [list(map(str, df.columns)), ["-"] * len(df.columns), *df.astype(str).values.tolist()]
    return "\n".join(f"| {' | '.join(row)} |" for row in rows)


years_loaded = pd.DataFrame(
    {
        "years loaded": [len(years) for years in years_by_tile.values()],
        "years expected": len(YEARS),
        "years": [f"{min(years)}-{max(years)}" if years else "" for years in years_by_tile.values()],
    },
    index=[f"{x:03}_{y:03}" for x, y in years_by_tile],
)
settings = pd.Series(
    {
        "flip_rate": f"more than {FLIP_THRESHOLD} year-to-year changes",
        "flicker": f"A -> B -> A where B lasts at most {FLICKER_MAX_YEARS} years",
        "implausible": "any transition in the implausible list below",
        "low_modal_share": f"modal class covers less than {MODAL_THRESHOLD:.0%} of years",
        "many_classes": f"more than {DISTINCT_THRESHOLD} distinct classes",
    },
    name="flagged when",
)
implausible_list = ", ".join(f"{classes_flipped[a]} -> {classes_flipped[b]}" for a, b in IMPLAUSIBLE)
unstable_figures = "\n\n".join(
    f"![Unstable pixels {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in unstable_pngs.items()
)
flicker_figures = "\n\n".join(
    f"![Flicker classes {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in flicker_pngs.items()
)
implausible_figures = "\n\n".join(
    f"![Implausible transitions {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in implausible_pngs.items()
)
area_figures = "\n\n".join(f"![Class area {tile[0]:03}_{tile[1]:03}]({png.name})" for tile, png in area_pngs.items())
few_years_note = (
    f"\n**Warning:** some tiles have fewer than {len(YEARS)} years of LULC, so the flip rate and flicker tests "
    "are weak.\n"
    if (years_loaded["years loaded"] < len(YEARS)).any()
    else ""
)

report = f"""# LULC temporal stability report: version {LULC_VERSION}

- LULC: {LULC_PREFIX} version {LULC_VERSION}
- Tiles: {", ".join(years_loaded.index)}
{few_years_note}
## Years loaded

{markdown_table(years_loaded)}

## Tests

{markdown_table(settings.to_frame())}

Implausible transitions: {implausible_list}.

## Unstable pixels per tile (count)

A pixel can fail several tests, so the test columns do not sum to `any`. `valid` is the number of pixels tested.

{markdown_table(summary)}

## Unstable pixels per tile (% of valid pixels)

{markdown_table(pct)}

## Unstable pixel map

{unstable_figures}

## Flicker classes

Each A -> B -> A event is found at the first year the pixel returns to A.

{flicker_figures}

## Implausible transitions

{implausible_figures}

## Class area time series

Year-to-year jumps that coincide with Landsat sensor changes or low GeoMAD observation counts are artefacts, not
land change.

{area_figures}

## Caveats

- Pixels with nodata in any year are not tested.
- Class areas use the EPSG:3832 pixel size, which is not equal area.
"""
report_path = REPORT_DIR / "temporal_stability.md"
report_path.write_text(report)
print(f"Wrote {report_path}")